# Question 4
Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove exact duplicates, and sort by order_date. 

In [0]:
df = spark.read.csv(
    "/Volumes/cyntexa_dev/sales/my_volume/ecommerce2.csv",
    header=True,
    inferSchema=True
)


In [0]:
from pyspark.sql.functions import *
df = df.dropna(how='all')\
    .dropDuplicates()\
    .withColumn('customer_name', when(col('customer_name').isNull(), 'Unknow').otherwise(col('customer_name')))\
    .fillna({
        'category': 'Unknown_Category'
    })\
    .dropna(how='any' , subset=['order_date'])\
    .sort(col('order_date').desc())
    


# Question 5
Perform an aggregation (revenue by category or region) and a join against a second small reference table (e.g., customers or regions). 

In [0]:
from pyspark.sql.window import * 
from pyspark.sql.functions import * 

df_customer = spark.read.table('samples.tpch.customer')
df_nation = spark.read.table('samples.tpch.nation')
df_orders = spark.read.table('samples.tpch.orders')
df_region = spark.read.table('samples.tpch.region')

df = df_customer.join(df_nation, df_customer['c_nationkey'] == df_nation['n_nationkey']).join(df_orders, df_customer['c_custkey'] == df_orders['o_custkey']).join(df_region, df_nation['n_regionkey'] == df_region['r_regionkey']).filter(df_orders['o_orderstatus'] == 'F')

customer_wise_region_amount =  df.select('c_custkey', 'c_name', 'o_orderstatus', 'o_totalprice' , "r_regionkey",'r_name')

# Customer-wise total within region
region_wise_total = (
    customer_wise_region_amount
    .groupBy(
        'r_regionkey',
        'r_name',
        'c_custkey',
        'c_name'
    )
    .agg(
        sum('o_totalprice').alias('region_wise_total')
    )
)

# Rank customers within each region
wind = Window.partitionBy("r_regionkey").orderBy(
    col("region_wise_total").desc()
)

region_wise_total = (
    region_wise_total
    .withColumn("rnk", rank().over(wind))
    .filter(col("rnk") <= 5)
)




region_wise_total.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("cyntexa_dev.analytics.region_wise_top5_customers")



### Points covered in this question

1. **Read multiple reference tables** — Customer, Nation, Orders, Region.
2. **Performed joins** between customer, nation, orders, and region tables.
3. **Filtered completed orders** using `o_orderstatus = 'F'`.
4. **Selected required columns** for customer, order, and region analysis.
5. **Performed aggregation** using `groupBy()` and `sum()`.
6. **Calculated customer-wise total revenue within each region.**
7. **Used Window Function** to rank customers within each region.
8. **Selected Top 5 customers per region.**
9. **Stored the final analytical result** as a Delta table in the Analytics layer.


# Question 6
Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request describing what changed and why. 

This Change Will be pushed to feature branch